# 11 DAC0-ADC0 Coherent Scope/Spectrum

Stage 8 entry point for coherent DAC0 to ADC0 single-tone preview with full-rate 245.76 MS/s metadata.


In [ ]:
from pathlib import Path
import asyncio
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/home/xilinx/jupyter_notebooks/t510_fengine'),
    Path('/home/xilinx/t510_fengine_bringup'),
    Path('/home/astrolab/demo-ant'),
]
PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'overlay' / 't510_fengine.bit').exists() and (root / 'python' / 't510_fengine.py').exists():
        PROJECT_ROOT = root
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find overlay/t510_fengine.bit and python/t510_fengine.py')
sys.path.insert(0, str(PROJECT_ROOT))
from python.t510_fengine import T510FEngine

BITFILE = PROJECT_ROOT / 'overlay' / 't510_fengine.bit'
EXPECTED_CORE_VERSION = 0x00010006
COLORS = ['#0b5cad', '#c45200', '#217a3b', '#b3261e', '#6f42c1', '#5f6368', '#008b8b', '#9a7b00']
state = {'core': None, 'live': False, 'task': None, 'last_cfg': None, 'last_error': None}

download_widget = W.Checkbox(value=True, description='Download bitstream')
center_widget = W.FloatText(value=1500.0, description='Center MHz', style={'description_width': '130px'})
bw_widget = W.FloatSlider(value=100.0, min=5.0, max=120.0, step=5.0, description='BW MHz', readout_format='.0f', style={'description_width': '130px'})
tone_widget = W.FloatSlider(value=20.0, min=0.25, max=50.0, step=0.25, description='Tone MHz', readout_format='.2f', style={'description_width': '130px'})
amplitude_widget = W.IntSlider(value=2048, min=0, max=8192, step=128, description='Amplitude', style={'description_width': '130px'})
phase_widget = W.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='Phase deg/ch', readout_format='.0f', style={'description_width': '130px'})
channels_widget = W.IntSlider(value=1, min=1, max=8, step=1, description='Visible ch', style={'description_width': '130px'})
samples_widget = W.Dropdown(options=[256, 512, 1024], value=512, description='Samples', style={'description_width': '130px'})
interval_widget = W.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description='Refresh s', readout_format='.1f', style={'description_width': '130px'})
timeout_widget = W.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description='Timeout s', readout_format='.1f', style={'description_width': '130px'})
load_button = W.Button(description='Load overlay', button_style='primary')
init_button = W.Button(description='Init RFDC', button_style='success')
apply_button = W.Button(description='Apply tone/NCO')
capture_button = W.Button(description='Capture once')
start_button = W.Button(description='Start live', button_style='success')
stop_button = W.Button(description='Stop live', button_style='warning')
status_out = W.Output(layout={'border': '1px solid #ccc', 'padding': '8px'})
plot_out = W.Output(layout={'border': '1px solid #ccc', 'padding': '8px'})

def core():
    if state['core'] is None:
        raise RuntimeError('Load overlay first')
    return state['core']

def visible_mask():
    return (1 << int(channels_widget.value)) - 1

def load_core(_=None):
    state['core'] = T510FEngine(str(BITFILE), download=bool(download_widget.value))
    with status_out:
        clear_output(wait=True)
        print(f'Project root: {PROJECT_ROOT}')
        print(f'Loaded {BITFILE}')

def apply_settings(start_if_needed=False):
    c = core()
    center_hz = float(center_widget.value) * 1_000_000.0
    bw_hz = float(bw_widget.value) * 1_000_000.0
    tone_hz = float(tone_widget.value) * 1_000_000.0
    if start_if_needed:
        c.stop()
        time.sleep(0.05)
        c.configure_clock(ref='tcxo_10mhz')
        c.set_adc_active_mask(0x0003)
        c.set_sync_mode('free_run')
        c.set_mode('spec')
        c.configure_rfdc(fs_adc=245_760_000, f_center=center_hz, bandwidth=bw_hz, decimation=20)
    nco = c.configure_rfdc_center_frequency(center_hz, bandwidth_hz=bw_hz, require=True)
    tone = c.configure_dac_tone_bank(
        freq_hz=tone_hz,
        amplitude=int(amplitude_widget.value),
        phase_deg_per_channel=float(phase_widget.value),
        enable_mask=visible_mask(),
        dac_sample_rate_hz=245_760_000.0,
    )
    epoch = c.reset_dac_phase()
    if start_if_needed:
        c.start()
    state['last_cfg'] = {'nco': nco, 'tone': tone, 'epoch': epoch, 'center_hz': center_hz, 'bw_hz': bw_hz}
    return state['last_cfg']

def init_rfdc(_=None):
    cfg = apply_settings(start_if_needed=True)
    with status_out:
        clear_output(wait=True)
        print(f'Initialized center={cfg["center_hz"] / 1e6:.3f} MHz bw={cfg["bw_hz"] / 1e6:.1f} MHz DAC epoch={cfg["epoch"]}')

def apply_only(_=None):
    cfg = apply_settings(start_if_needed=False)
    with status_out:
        clear_output(wait=True)
        print(f'Applied tone={cfg["tone"]["freq_hz"] / 1e6:.3f} MHz phase_step=0x{cfg["tone"]["phase_step"]:08x} epoch={cfg["epoch"]}')

def interp_peak_hz(freq_hz, power, peak_idx):
    if peak_idx <= 0 or peak_idx >= len(power) - 1:
        return float(freq_hz[peak_idx])
    alpha = np.log(max(float(power[peak_idx - 1]), 1.0))
    beta = np.log(max(float(power[peak_idx]), 1.0))
    gamma = np.log(max(float(power[peak_idx + 1]), 1.0))
    denom = alpha - 2.0 * beta + gamma
    if abs(denom) < 1e-12:
        return float(freq_hz[peak_idx])
    delta = 0.5 * (alpha - gamma) / denom
    delta = float(np.clip(delta, -1.0, 1.0))
    return float(freq_hz[peak_idx] + delta * (freq_hz[1] - freq_hz[0]))

def compute_spectrum(preview):
    sample_rate = float(preview['sample_rate_hz'])
    count = int(preview['count'])
    signed_freq = np.fft.fftshift(np.fft.fftfreq(count, d=1.0 / sample_rate))
    bw_hz = float(bw_widget.value) * 1_000_000.0
    passband = np.abs(signed_freq) <= bw_hz / 2.0
    result = {'freq_hz': signed_freq, 'power': {}, 'peaks': {}}
    ref_phase = None
    for ch, iq in preview['iq'].items():
        arr = np.asarray(iq, dtype=np.float64)
        z = (arr[:, 0] + 1j * arr[:, 1]) - np.mean(arr[:, 0] + 1j * arr[:, 1])
        fft = np.fft.fftshift(np.fft.fft(z * np.hanning(count)))
        pwr = np.abs(fft) ** 2
        masked = np.where(passband, pwr, 0.0)
        peak_idx = int(np.argmax(masked))
        peak_hz = interp_peak_hz(signed_freq, masked, peak_idx)
        phase = float(np.angle(fft[peak_idx], deg=True))
        if ch == 0:
            ref_phase = phase
        result['power'][ch] = pwr
        result['peaks'][ch] = {'peak_idx': peak_idx, 'raw_peak_mhz': signed_freq[peak_idx] / 1e6, 'peak_mhz': peak_hz / 1e6, 'phase_deg': phase, 'peak_power': float(pwr[peak_idx])}
    if ref_phase is not None:
        for pk in result['peaks'].values():
            delta = pk['phase_deg'] - ref_phase
            while delta > 180.0:
                delta -= 360.0
            while delta <= -180.0:
                delta += 360.0
            pk['delta_phase_deg'] = delta
    return result

def render(preview, spectrum, elapsed):
    status = core().read_status()
    cfg = state.get('last_cfg') or {}
    with status_out:
        clear_output(wait=True)
        print(f'CORE_VERSION=0x{status["core_version"]:08x} expected=0x{EXPECTED_CORE_VERSION:08x} streaming={status["streaming"]}')
        print(f'sample0={preview["sample0"]} count={preview["count"]} preview_rate={preview["sample_rate_hz"]} Hz axis_rate={preview.get("axis_beat_rate_hz", 0)} Hz elapsed={elapsed:.3f}s')
        print(f'center={float(center_widget.value):.3f} MHz bw={float(bw_widget.value):.1f} MHz tone={float(tone_widget.value):.3f} MHz dac_epoch={status.get("dac_phase_epoch", 0)}')
        print(f'ADC port mask current=0x{status["rfdc_current_valid_mask"]:04x} required=0x0003 UDP_DRY_RUN={status["udp_dry_run"]} QSFP_LINK_UP={status["qsfp_link_up"]}')
        for ch in preview['inputs']:
            pk = spectrum['peaks'][ch]
            label = 'physical DAC0-ADC0' if ch == 0 else 'digital/control only'
            print(f'CH{ch}: {label} peak={pk["peak_mhz"]:.3f} MHz phase={pk["phase_deg"]:.1f} deg delta={pk.get("delta_phase_deg", 0.0):.1f} deg')

    with plot_out:
        clear_output(wait=True)
        fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 4.8))
        t_us = np.arange(int(preview['count']), dtype=np.float64) / float(preview['sample_rate_hz']) * 1e6
        for ch in preview['inputs']:
            arr = np.asarray(preview['iq'][ch])
            color = COLORS[ch % len(COLORS)]
            ax0.plot(t_us, arr[:, 0], color=color, lw=1.0, label=f'CH{ch} I')
        ax0.set_title('Coherent scope')
        ax0.set_xlabel('time (us)')
        ax0.set_ylabel('ADC code')
        ax0.grid(True, alpha=0.3)
        ax0.legend(fontsize=8)
        freq_mhz = spectrum['freq_hz'] / 1e6
        for ch in preview['inputs']:
            pwr = np.asarray(spectrum['power'][ch], dtype=np.float64)
            ax1.plot(freq_mhz, 10.0 * np.log10(np.maximum(pwr, 1.0)), color=COLORS[ch % len(COLORS)], lw=1.0, label=f'CH{ch}')
            ax1.axvline(spectrum['peaks'][ch]['peak_mhz'], color=COLORS[ch % len(COLORS)], alpha=0.2)
        half_bw = float(bw_widget.value) / 2.0
        ax1.set_xlim(-half_bw, half_bw)
        ax1.axvline(float(tone_widget.value), color='#111111', lw=1.0, ls='--', alpha=0.35)
        ax1.axvline(-float(tone_widget.value), color='#111111', lw=1.0, ls=':', alpha=0.25)
        ax1.set_title('Coherent spectrum')
        ax1.set_xlabel('baseband frequency (MHz)')
        ax1.set_ylabel('power (dB)')
        ax1.grid(True, alpha=0.3)
        ax1.legend(fontsize=8)
        plt.tight_layout()
        display(fig)
        plt.close(fig)

def capture_once(_=None):
    t0 = time.monotonic()
    preview = core().capture_preview(n=int(samples_widget.value), input_mask=visible_mask(), timeout=float(timeout_widget.value))
    spectrum = compute_spectrum(preview)
    render(preview, spectrum, time.monotonic() - t0)

async def live_loop():
    state['live'] = True
    while state['live']:
        try:
            capture_once()
            state['last_error'] = None
        except Exception as exc:
            state['last_error'] = exc
            with status_out:
                clear_output(wait=True)
                print(f'ERROR: {exc}')
            state['live'] = False
            break
        await asyncio.sleep(float(interval_widget.value))

def start_live(_=None):
    if state.get('task') is None or state['task'].done():
        state['task'] = asyncio.create_task(live_loop())

def stop_live(_=None):
    state['live'] = False

load_button.on_click(load_core)
init_button.on_click(init_rfdc)
apply_button.on_click(apply_only)
capture_button.on_click(capture_once)
start_button.on_click(start_live)
stop_button.on_click(stop_live)

display(W.VBox([
    W.HBox([load_button, init_button, apply_button, capture_button, start_button, stop_button, download_widget]),
    W.HBox([center_widget, bw_widget, tone_widget, amplitude_widget]),
    W.HBox([phase_widget, channels_widget, samples_widget, interval_widget, timeout_widget]),
    status_out,
    plot_out,
]))
